# **Keşifçi Veri Analizi (Exploratory Data Analysis - EDA)**
## World GDP Dataset (IMF, 1980–2023)

### Kütüphanelerin Yüklenmesi

In [ ]:
# temel veri işleme kütüphaneleri
import pandas as pd
import numpy as np

# görselleştirme kütüphaneleri
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# istatistik kütüphaneleri
from scipy import stats
import warnings

# ayarlar
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Grafik stili ayarları
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('Tüm kütüphaneler başarıyla yüklendi')
print(f'Pandas version: {pd.__version__}')
print(f'Numpy version: {np.__version__}')
print(f'Seaborn version: {sns.__version__}')

## Veri Yükleme

In [ ]:
# CSV dosyasını yükle — son 4 satır footer (©IMF, boş satırlar) olduğu için atla
df = pd.read_csv('World_GDP_Dataset.csv', skipfooter=4, engine='python')

# Sütun adını temizle (BOM karakteri olabilir)
df.columns = [col.strip() for col in df.columns]
df.rename(columns={df.columns[0]: 'Country'}, inplace=True)

print(' World GDP Verisi')
print('=' * 50)
print(f'Boyut : {df.shape[0]} satır, {df.shape[1]} sütun')
print('=' * 50)
print(f'Bellek kullanımı: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')

### İlk Bakış

In [ ]:
# ilk 5 satır
df.head()

In [ ]:
# son 5 satır
df.tail()

In [ ]:
# rastgele 5 satır
df.sample(5)

### Veri Tipleri ve Temel İstatistikler

In [ ]:
print('Veri tipleri')
print('=' * 40)
dtype_df = pd.DataFrame({
    'sütun': df.columns,
    'veri tipi': df.dtypes.values,
    'null değeri': df.isnull().sum().values,
    'null oranı(%)': (df.isnull().sum().values / len(df) * 100).round(2),
    'unique': df.nunique().values
})
print(dtype_df.to_string(index=False))

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
# Yıl sütunlarını belirle
yil_sutunlari = [col for col in df.columns if col.isdigit()]

# Sayısal sütunlara ait temel istatistikler (yıllar)
df[yil_sutunlari].describe().T

In [ ]:
# Kategorik sütunlara ait istatistikler (Country)
df[['Country']].describe()

In [ ]:
# Değişken türlerini ayır
sayisal_sutunlar = df.select_dtypes(include=['int64', 'float64']).columns
kategorik_sutunlar = df.select_dtypes(include=['object', 'category']).columns

print(f'Sayısal sütun sayısı: {len(sayisal_sutunlar)}')
print(f'Kategorik sütun sayısı: {len(kategorik_sutunlar)}')
print(f'\nKategorik sütunlar: {list(kategorik_sutunlar)}')

### Eksik Veri Analizi

In [ ]:
# Sıfır değerleri gerçek eksik veri gibi davranıyor (1980'lerde birçok ülke için 0)
# Önce klasik NaN eksik analizi
eksik = df.isnull().sum()
eksik_orani = (eksik / len(df) * 100).round(2)
eksik_df = pd.concat([eksik, eksik_orani], axis=1)
eksik_df.columns = ['eksik_sayı', 'eksik_oran']
eksik_df.sort_values('eksik_oran', ascending=False)

In [ ]:
# Sıfır değerleri de eksik veri sayalım (GDP 0 olamaz → gerçek veri eksikliği)
sifir_sayisi = (df[yil_sutunlari] == 0).sum()
sifir_orani = (sifir_sayisi / len(df) * 100).round(2)

sifir_df = pd.DataFrame({
    'sıfır_sayı': sifir_sayisi,
    'sıfır_oran(%)': sifir_orani
})
print('Yıllara Göre Sıfır (Eksik) Veri Oranı:')
print(sifir_df.sort_values('sıfır_oran(%)', ascending=False).head(20).to_string())

In [ ]:
# Eksik veri ısı haritası — her yıl için sıfır olan ülke oranı
fig, ax = plt.subplots(figsize=(16, 6))

sifir_yuzde = [(df[yil] == 0).sum() / len(df) * 100 for yil in yil_sutunlari]
ax.bar(yil_sutunlari, sifir_yuzde, color='steelblue', alpha=0.8)
ax.set_title('Yıllara Göre Eksik (Sıfır) GDP Verisi Oranı (%)', fontsize=14)
ax.set_xlabel('Yıl')
ax.set_ylabel('Eksik Veri Oranı (%)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### Eksik Veri Temizleme Stratejisi

In [ ]:
# Sıfır değerleri NaN ile değiştir (gerçek eksik veri olarak işaretle)
df_temiz = df.copy()
df_temiz[yil_sutunlari] = df_temiz[yil_sutunlari].replace(0, np.nan)

print('Sıfırlar NaN olarak işaretlendi.')
print(f'Toplam NaN sayısı: {df_temiz[yil_sutunlari].isnull().sum().sum()}')

In [ ]:
# Strateji: Her ülke için eksik yılları doğrusal interpolasyon ile doldur
# (zaman serisi verisi olduğu için interpolasyon medyandan daha mantıklı)
df_temiz[yil_sutunlari] = df_temiz[yil_sutunlari].apply(
    lambda row: row.interpolate(method='linear', limit_direction='both'), axis=1
)

print('Eksik veri stratejisi: Doğrusal interpolasyon uygulandı.')
print(f'Kalan NaN sayısı: {df_temiz[yil_sutunlari].isnull().sum().sum()}')

In [ ]:
# Önce / Sonra karşılaştırması
print('ÖNCESİ (ham veri):')
print(f'  Sıfır sayısı: {(df[yil_sutunlari] == 0).sum().sum()}')
print()
print('SONRASI (temizlenmiş veri):')
print(f'  NaN sayısı  : {df_temiz[yil_sutunlari].isnull().sum().sum()}')
print(f'  Sıfır sayısı: {(df_temiz[yil_sutunlari] == 0).sum().sum()}')

## Görselleştirme ve Keşif

In [ ]:
# 2023 yılında en yüksek GDP'ye sahip 15 ülke
top15_2023 = df_temiz[['Country', '2023']].sort_values('2023', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top15_2023, x='2023', y='Country', ax=ax, palette='husl')
ax.set_title("2023 Yılında En Yüksek GDP'ye Sahip 15 Ülke (Milyar $)", fontsize=14)
ax.set_xlabel('GDP (Milyar $)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 2023 yılında en düşük GDP'ye sahip 15 ülke (NaN olmayanlar arasından)
bottom15_2023 = df_temiz[['Country', '2023']].dropna().sort_values('2023').head(15)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=bottom15_2023, x='2023', y='Country', ax=ax, palette='rocket')
ax.set_title("2023 Yılında En Düşük GDP'ye Sahip 15 Ülke (Milyar $)", fontsize=14)
ax.set_xlabel('GDP (Milyar $)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Dünya toplam GDP'sinin yıllar içindeki değişimi
dunya_gdp = df_temiz[yil_sutunlari].sum()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(yil_sutunlari, dunya_gdp.values, color='steelblue', linewidth=2.5, marker='o', markersize=4)
ax.set_title('Dünya Toplam GDP (1980–2023)', fontsize=14)
ax.set_xlabel('Yıl')
ax.set_ylabel('Toplam GDP (Milyar $)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Seçili büyük ekonomilerin GDP trendi
ulkeler = ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom']

fig, ax = plt.subplots(figsize=(14, 7))
for ulke in ulkeler:
    satir = df_temiz[df_temiz['Country'] == ulke]
    if not satir.empty:
        ax.plot(yil_sutunlari, satir[yil_sutunlari].values[0], label=ulke, linewidth=2)

ax.set_title('Büyük Ekonomilerde GDP Trendi (1980–2023)', fontsize=14)
ax.set_xlabel('Yıl')
ax.set_ylabel('GDP (Milyar $)')
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Türkiye'nin GDP trendi
fig, ax = plt.subplots(figsize=(14, 6))
turkiye = df_temiz[df_temiz['Country'] == 'Türkiye']

# IMF verisinde 'Turkey' veya 'Türkiye' olabilir, ikisini de dene
if turkiye.empty:
    turkiye = df_temiz[df_temiz['Country'] == 'Turkey']

if not turkiye.empty:
    ax.plot(yil_sutunlari, turkiye[yil_sutunlari].values[0],
            color='#E30A17', linewidth=2.5, marker='o', markersize=4)
    ax.set_title("Türkiye GDP Trendi (1980–2023)", fontsize=14)
else:
    ax.set_title('Türkiye verisi bulunamadı — lütfen Country adını kontrol edin')

ax.set_xlabel('Yıl')
ax.set_ylabel('GDP (Milyar $)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# 2023 GDP dağılımı — histogram (normal ve log ölçek)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gdp_2023 = df_temiz['2023'].dropna()

# Normal ölçek
axes[0].hist(gdp_2023, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('2023 GDP Dağılımı (Normal Ölçek)')
axes[0].set_xlabel('GDP (Milyar $)')
axes[0].set_ylabel('Ülke Sayısı')

# Log ölçek (sağa çarpık dağılım için)
axes[1].hist(np.log1p(gdp_2023), bins=30, color='coral', edgecolor='white')
axes[1].set_title('2023 GDP Dağılımı (Log Ölçek)')
axes[1].set_xlabel('log(GDP)')
axes[1].set_ylabel('Ülke Sayısı')

plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon matrisi — seçili yıllar
secili_yillar = ['1990', '1995', '2000', '2005', '2010', '2015', '2020', '2023']
corr_matrix = df_temiz[secili_yillar].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Seçili Yıllar Arası GDP Korelasyon Matrisi', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot — seçili yıllar için GDP dağılımı
fig, ax = plt.subplots(figsize=(14, 7))
df_temiz[secili_yillar].boxplot(ax=ax)
ax.set_title('Seçili Yıllarda GDP Dağılımı (Boxplot)', fontsize=14)
ax.set_xlabel('Yıl')
ax.set_ylabel('GDP (Milyar $)')
plt.tight_layout()
plt.show()

### Aykırı Değer Analizi

In [ ]:
# 2023 yılı için IQR yöntemi ile aykırı değer tespiti
Q1 = df_temiz['2023'].quantile(0.25)
Q3 = df_temiz['2023'].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

aykiri = df_temiz[(df_temiz['2023'] < alt_sinir) | (df_temiz['2023'] > ust_sinir)]

print('IQR Yöntemi — 2023 GDP Aykırı Değer Analizi')
print(f'Q1          : {Q1:.2f} Milyar $')
print(f'Q3          : {Q3:.2f} Milyar $')
print(f'IQR         : {IQR:.2f}')
print(f'Alt sınır   : {alt_sinir:.2f}')
print(f'Üst sınır   : {ust_sinir:.2f}')
print(f'\nAykırı değer sayısı: {len(aykiri)}')
print('\nAykırı değer olan ülkeler:')
print(aykiri[['Country', '2023']].sort_values('2023', ascending=False).to_string(index=False))

In [ ]:
# Z-score yöntemi ile aykırı değer analizi (|z| > 3)
gdp_2023_temiz = df_temiz['2023'].dropna()
z_skorlari = np.abs(stats.zscore(gdp_2023_temiz))

aykiri_z = df_temiz.loc[gdp_2023_temiz.index[z_skorlari > 3], ['Country', '2023']]
print('Z-Score Yöntemi (|z| > 3) — Aykırı Değerler:')
print(aykiri_z.sort_values('2023', ascending=False).to_string(index=False))

### Büyüme Hızı Analizi

In [ ]:
# 2000-2023 arası GDP büyümesi (kat artış)
df_temiz['buyume_2000_2023'] = df_temiz['2023'] / df_temiz['2000']

top10_buyume = df_temiz[['Country', '2000', '2023', 'buyume_2000_2023']].dropna()
top10_buyume = top10_buyume[top10_buyume['2000'] > 1]  # çok küçük ülkeleri çıkar
top10_buyume = top10_buyume.sort_values('buyume_2000_2023', ascending=False).head(10)

print('2000-2023 Arası En Hızlı Büyüyen 10 Ekonomi:')
print(top10_buyume.to_string(index=False))

In [ ]:
# Büyüme görselleştirmesi
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top10_buyume, x='buyume_2000_2023', y='Country', ax=ax, palette='viridis')
ax.set_title('2000–2023 Arası En Hızlı Büyüyen 10 Ekonomi (Kat Artış)', fontsize=14)
ax.set_xlabel('Büyüme Katsayısı (2023 GDP / 2000 GDP)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### GDP Pay Analizi

In [ ]:
# 2023 yılında ülkelerin dünya GDP'sindeki payı (ilk 10)
toplam_2023 = df_temiz['2023'].sum()
df_temiz['pay_2023'] = (df_temiz['2023'] / toplam_2023 * 100).round(2)

top10_pay = df_temiz[['Country', '2023', 'pay_2023']].dropna().sort_values('pay_2023', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    top10_pay['pay_2023'],
    labels=top10_pay['Country'],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.85
)
ax.set_title('2023 Dünya GDP Payı — İlk 10 Ülke (%)', fontsize=14)
plt.tight_layout()
plt.show()

### Temizlenmiş Veri — Son Durum

In [ ]:
print('Temizlenmiş veri özeti:')
df_temiz.info()

In [ ]:
kategorik_sutunlar = df_temiz.select_dtypes(include=['object', 'category']).columns
kategorik_sutunlar

In [ ]:
df_temiz.info()